In [ ]:
# setup: rebuild everything this notebook needs from files
%load_ext autoreload
%autoreload 2
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from src.data import build_daily, load_loads
from src.signals import build_signals, load_events
from src.splits import load_splits

sp = load_splits("../data/processed/splits.json")
daily = build_daily(load_loads())
train = daily.loc[:sp["train"][1]]
valid = daily.loc[sp["valid"][0]:sp["valid"][1]]
signals = build_signals(daily, load_events())
features = pd.read_csv("../data/processed/features.csv", index_col=0, parse_dates=True)
print(len(train), len(valid), signals.shape)

928 84 (1091, 3)


In [ ]:
# daily signals option 1: build the signals
from src.signals import build_signals, load_events

signals = build_signals(daily, load_events())
print(signals.shape, signals.index.min().date())
print(signals.loc[:train.index.max()].describe().loc[["mean", "std"]].round(3))

(1091, 3) 2022-01-06
      volume  late_rate  detention
mean  77.944      0.444    106.740
std    8.561      0.055      8.732


In [ ]:
# daily signals option 2: why the first days are dropped
ev = load_events()
d = ev[ev["event_type"] == "Delivery"]
print(d.groupby(d["actual_datetime"].dt.normalize()).size().iloc[:8].tolist())

[10, 49, 77, 65, 80, 75, 74, 83]


In [ ]:
# anomaly injection option 1: inject into a copy of the validation signals
from src.anomalies import inject

val_sig = signals.loc[valid.index.min():valid.index.max()]
injected, labels42 = inject(val_sig, seed=42)
print(labels42.to_string(index=False))
print(val_sig.equals(signals.loc[val_sig.index]))
w = labels42.iloc[0]
print(val_sig.loc[w.start:w.end, "volume"].round(0).tolist(), injected.loc[w.start:w.end, "volume"].round(1).tolist())

   signal      start        end
   volume 2024-07-24 2024-07-28
   volume 2024-09-16 2024-09-20
late_rate 2024-08-21 2024-08-24
late_rate 2024-09-08 2024-09-11
detention 2024-08-20 2024-08-24
detention 2024-09-23 2024-09-27
True
[75.0, 70.0, 90.0, 84.0, 70.0] [60.0, 56.0, 72.0, 67.2, 56.0]


In [ ]:
# anomaly labels option 1: labels for 20 draws, saved on their own
from src.anomalies import make_all_labels

all_labels = make_all_labels(val_sig, range(42, 62))
all_labels.to_csv("../data/processed/anomaly_labels.csv", index=False)
print(all_labels.shape, all_labels["seed"].nunique())

(120, 4) 20


In [ ]:
# anomaly labels option 2: labels are not in any feature table
print(set(all_labels.columns) & set(features.columns))

set()


In [ ]:
# anomaly method option 1: the baseline the detector measures against
from src.detect import fit_baseline

base = fit_baseline(signals.loc[:train.index.max()])
print({k: {a: round(b, 3) for a, b in v.items()} for k, v in base.items()})

{'center': {'volume': 78.0, 'late_rate': 0.446, 'detention': 106.761}, 'scale': {'volume': 8.896, 'late_rate': 0.055, 'detention': 8.443}}


In [ ]:
# anomaly thresholds option 1: fit on clean training days
from src.detect import find_alerts, fit_baseline, fit_thresholds, score_signals

train_sig = signals.loc[:train.index.max()]
base = fit_baseline(train_sig)
train_scores = score_signals(train_sig, base).dropna()
thresholds = fit_thresholds(train_scores)
print({k: round(v, 2) for k, v in thresholds.items()})
print(len(find_alerts(train_scores, thresholds)), train_scores.size)

{'volume': 1.26, 'late_rate': 1.26, 'detention': 1.35}
30 2763


In [ ]:
# anomaly thresholds option 2: freeze the detector settings
from src.artifact import write_json

write_json("../reports/anomaly_thresholds.json", {
    "frozen": "2026-11-28",
    "quantile": 0.99,
    "roll_days": 3,
    "training_period": [str(train_sig.index.min().date()), str(train_sig.index.max().date())],
    "baseline": base,
    "thresholds": thresholds,
})
print(train_sig.index.min().date(), train_sig.index.max().date())

2022-01-06 2024-07-16


In [ ]:
# anomaly evaluation option 1: the frozen detector on 20 injected draws
import json
from pathlib import Path
from src.anomaly_eval import evaluate_all

frozen = json.loads(Path("../reports/anomaly_thresholds.json").read_text())
base, thresholds = frozen["baseline"], frozen["thresholds"]
train_sig = signals.loc[:sp["train"][1]]
val_sig = signals.loc[sp["valid"][0]:sp["valid"][1]]
res = evaluate_all(train_sig, val_sig, base, thresholds, range(42, 62))
g = res.groupby("signal")[["alerts", "correct", "false_alerts", "windows", "detected", "clean_days"]].sum()
g["precision"] = (g["correct"] / g["alerts"]).round(2)
g["recall"] = (g["detected"] / g["windows"]).round(2)
print(g[["alerts", "correct", "windows", "detected", "precision", "recall"]])
tot = g.sum()
print(round(tot["correct"] / tot["alerts"], 2), round(tot["detected"] / tot["windows"], 2))

In [ ]:
# anomaly evaluation option 2: false alarms per 100 days without an anomaly
g["false_per_100_days"] = (100 * g["false_alerts"] / g["clean_days"]).round(2)
print(g[["false_alerts", "clean_days", "false_per_100_days"]])